검증 세트(Validation Set)

In [1]:
import pandas as pd
wine = pd.read_csv('https://bit.ly/wine_csv_data')

In [2]:
import numpy as np

data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

In [3]:
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(data, target, test_size=0.2, random_state=42)

In [4]:
sub_input, val_input, sub_target, val_target = train_test_split(train_input, train_target, test_size=0.2, random_state=42)

In [7]:
print(sub_input.shape, val_input.shape)

(4157, 3) (1040, 3)


In [9]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)
print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))

0.9971133028626413
0.864423076923077


과대적합. 매개변수 조정 필요

교차검증(Cross Validation)

sklearn의 cross_validate() 교차검증 함수 사용

먼저 평가할 모델 객체를 첫 번째 매개변수로 전달.

검증 세트를 떼어 내지 않고 훈련 세트 전체를 cross_validate() 함수에 전달

In [11]:
from sklearn.model_selection import cross_validate

scores = cross_validate(dt, train_input, train_target)
print(scores)

{'fit_time': array([0.01086879, 0.00823069, 0.00815916, 0.00790024, 0.00781417]), 'score_time': array([0.00154257, 0.00140238, 0.00132608, 0.00124168, 0.00121641]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


fit_time, score_time, test_score 키를 가진 딕셔너리를 반환

처음 2개의 키는 각각 모델을 훈련하는 시간과 검증하는 시간을 의미

기본적으로 5-폴드 교차 검증을 수행. cv 매개변수에서 폴드수를 변경 가능

**교차 검증의 최종 점수는 test_score 키에 담긴 5개의 점수를 평균하여 구할 수 있다.**

In [12]:
import numpy as np

print(np.mean(scores['test_score']))

0.855300214703487


주의할 점은 cross_validate()는 훈련세트를 섞어 폴드를 나누지 않는다.

만약 교차 검증을 할 때 훈련 세트를 섞으려면 분할기(Splitter)를 지정해야한다.

sklearn 분할기는 교차 검증에서 폴드를 어떻게 나눌지 결정해준다.

cross_validate() 함수는 기본적으로 **회귀의 경우 KFold 분할기, 분류의 경우 타겟 클래스를 골고루 나누기 위해 StratifiedKFold를 사용한다.**

In [13]:
from sklearn.model_selection import StratifiedKFold

scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
print(np.mean(scores['test_score']))

0.855300214703487


In [14]:
# 훈련 세트를 섞은 후 10-fold 교차 검증을 수행하는 경우
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_validate(dt, train_input, train_target, cv=splitter)
print(np.mean(scores['test_score']))

0.8574181117533719


### 하이퍼파라미터 튜닝
AutoML : 사람의 개입 없이 하이퍼파라미터 튜닝을 자동으로 수행하는 기술

sklearn의 GridSearchCV는 하이퍼파라미터 탐색과 교차 검증을 한 번에 수행

In [19]:
from sklearn.model_selection import GridSearchCV
params = {'min_impurity_decrease': [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]}

In [20]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)

GridSearchCV의 cv 매개변수 기본값은 5

따라서 min_inpurity_decrease 값을 바꿔가며 총 5번 실행 (5X5=25개의 모델 훈련)'

n_jobs 매개변수에서 병렬 실행에 사용할 CPU 코어 수를 지정하는 것이 좋다.

기본 값은 1이며 -1로 지정하면 시스템에 있는 모든 코어를 사용

In [21]:
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'min_impurity_decrease': [0.0001, 0.0002, 0.0003,
                                                   0.0004, 0.0005]})

In [23]:
# sklearn의 그리드 서치는 훈련이 끝나면 교차 검증 점수가 가장 높은 모델의 매개변수 조합으로 전체 train set에서 자동으로 다시 모델 훈련
# 이 모델은 gs 객체의 best_estimator_ 속성에 저장
dt = gs.best_estimator_
print(dt.score(train_input, train_target))

0.9615162593804117


In [24]:
# GrideSearch로 찾은 최적의 매개변수는 best_params_ 속성에 저장
print(gs.best_params_)

{'min_impurity_decrease': 0.0001}


In [25]:
# 각 매개변수에서 수행한 교차 검증의 평균 점수는 cv_result_ 속성의 'mean_test_score'에 저장
print(gs.cv_results_['mean_test_score'])

[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]


In [27]:
# 넘파이의 argmax() 함수를 사용하면 가장 큰 값의 인덱스 추출 가능
best_index = np.argmax(gs.cv_results_['mean_test_score'])
print(gs.cv_results_['params'][best_index])

{'min_impurity_decrease': 0.0001}


### 정리

1. 먼저 탐색할 매개변수 지정

2. 그 다음 train set에서 그리드 서치를 수행하여 최상의 평균 검증 점수가 나오는 매개변수 조합을 찾는다. 이 조합은 그리드 서치 객체에 저장된다.

3. 그리드 서치는 최상의 매개변수에서 전체 훈련세트를 사용해 최종 모델을 훈련한다. 이 모델도 그리드 서치 객체에 저장된다.


In [28]:
# 조금 더 복잡한 매개변수 조합 탐색

params = {'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001),
          'max_depth': range(5, 20, 1),
          'min_samples_split': range(2, 100, 10)
          }

In [29]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)
gs.fit(train_input, train_target)

/usr/local/lib/python3.10/dist-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': range(5, 20),
                         'min_impurity_decrease': array([0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008,
       0.0009]),
                         'min_samples_split': range(2, 100, 10)})

In [31]:
print(gs.best_params_)

{'max_depth': 14, 'min_impurity_decrease': 0.0004, 'min_samples_split': 12}


In [32]:
print(np.max(gs.cv_results_['mean_test_score']))

0.8683865773302731


### 랜덤 서치(Random Search)
매개변수의 값이 수치일 때 값의 범위나 간격을 미리 정하기 어려울 수 있음.

랜덤 서치는 매개변수 값의 목록을 전달하는 것이 아니라 매개변수를 샘플링할 수 있는 확률분포 객체를 전달

In [34]:
# uniform은 실수값을 뽑으며 randint는 정수값을 뽑음
# 모두 주어진 범위에서 고르게 값을 뽑음 (균등분포)
from scipy.stats import uniform, randint

In [35]:
rgen = randint(0, 10)
rgen.rvs(10)

array([7, 2, 3, 3, 7, 9, 3, 8, 9, 4])

In [36]:
np.unique(rgen.rvs(1000), return_counts=True)

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([112,  93, 109, 109,  86, 100, 118,  91,  85,  97]))

In [38]:
ugen = uniform(0, 1)
ugen.rvs(10)

array([0.70647729, 0.54134292, 0.00380033, 0.77916166, 0.99304525,
       0.50298669, 0.83356172, 0.22563903, 0.0983781 , 0.20959999])

In [39]:
params = {'min_impurity_decrease': uniform(0.001, 0.001),
          'max_depth': randint(20, 50),
          'min_samples_split': randint(2, 25),
          'min_samples_leaf': randint(1, 25)
          }

In [42]:
from sklearn.model_selection import RandomizedSearchCV
gs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params,
                        n_iter=100, n_jobs=-1, random_state=42)
gs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7e7d3393ffa0>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7e7d3393f820>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7e7d3393e050>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7e7d36196830>},
                   random_state=42)

In [43]:
print(gs.best_params_)

{'max_depth': 45, 'min_impurity_decrease': 0.001025419126744095, 'min_samples_leaf': 13, 'min_samples_split': 20}


In [44]:
print(np.max(gs.cv_results_['mean_test_score']))

0.8647301399274451


In [45]:
dt = gs.best_estimator_
print(dt.score(test_input, test_target))

0.8561538461538462
